# Retrieval Layer

Build a small physics-focused FAISS index from curated physics topic summaries, enrich from Wikipedia when available, and retrieve the most relevant chunks for the saved query.


In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import hashlib
import json
import math
import re
from typing import Any

import requests
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

from src.saferesponse_engine import logger
from src.saferesponse_engine.constants import CONFIG_FILE_PATH, PARAM_FILE_PATH, SCHEMA_FILE_PATH
from src.saferesponse_engine.utils.common import create_directories, read_yaml


In [ ]:
@dataclass(frozen=True)
class RetrievalConfig:
    root_dir: Path
    query_artifact_path: Path
    faiss_index_path: Path
    retrieval_output_path: Path
    embedding_model: str
    top_k: int
    chunk_size: int
    chunk_overlap: int
    num_articles: int
    min_score_threshold: float


class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.params = read_yaml(PARAM_FILE_PATH)
        self.schema = read_yaml(SCHEMA_FILE_PATH)

    def get_retrieval_layer_config(self) -> RetrievalConfig:
        config = self.config.retrieval_layer
        root_dir = Path(config.root_dir)
        create_directories([root_dir, Path(config.faiss_index_path), Path(config.retrieval_output_path).parent])

        return RetrievalConfig(
            root_dir=root_dir,
            query_artifact_path=Path(config.query_artifact_path),
            faiss_index_path=Path(config.faiss_index_path),
            retrieval_output_path=Path(config.retrieval_output_path),
            embedding_model=str(config.embedding_model),
            top_k=int(config.top_k),
            chunk_size=int(config.chunk_size),
            chunk_overlap=int(config.chunk_overlap),
            num_articles=int(config.num_articles),
            min_score_threshold=float(config.min_score_threshold),
        )


In [ ]:
class LocalHashEmbeddings(Embeddings):
    TOKEN_NORMALIZATION = {
        "impluse": "impulse",
        "momenta": "momentum",
    }

    def __init__(self, dimension: int = 256):
        self.dimension = dimension

    def _tokenize(self, text: str) -> list[str]:
        tokens = re.findall(r"[a-zA-Z0-9']+", text.lower())
        return [self.TOKEN_NORMALIZATION.get(token, token) for token in tokens]

    def _embed(self, text: str) -> list[float]:
        vector = [0.0] * self.dimension
        counts = Counter(self._tokenize(text))
        for token, weight in counts.items():
            digest = hashlib.sha256(token.encode("utf-8")).hexdigest()
            index = int(digest[:8], 16) % self.dimension
            vector[index] += float(weight)

        norm = math.sqrt(sum(value * value for value in vector))
        if norm == 0:
            return vector
        return [value / norm for value in vector]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)


class RetrievalLayer:
    PHYSICS_SUMMARIES = {
        "Physics": "Physics studies matter, energy, motion, force, and the laws that govern natural phenomena.",
        "Classical mechanics": "Classical mechanics describes the motion of bodies under forces using displacement, velocity, acceleration, mass, momentum, and energy.",
        "Newton's laws of motion": "Newton's laws relate force, mass, inertia, and acceleration, and they explain how motion changes when net external force acts on an object.",
        "Momentum": "Momentum is the product of mass and velocity. In an isolated system, total momentum is conserved during collisions and interactions.",
        "Impulse (physics)": "Impulse equals force multiplied by time and is equal to the change in momentum. A larger force acting over a short time can produce the same impulse as a smaller force over a longer time.",
        "Force": "Force is an interaction that can change an object's motion. According to Newton's second law, net force produces acceleration.",
        "Work (physics)": "Work is energy transferred when a force causes displacement. It depends on force, displacement, and the angle between them.",
        "Energy": "Energy is the capacity to do work. Common forms include kinetic energy, potential energy, thermal energy, and electromagnetic energy.",
        "Kinetic energy": "Kinetic energy is the energy associated with motion and depends on mass and the square of velocity.",
        "Thermodynamics": "Thermodynamics studies heat, work, temperature, internal energy, and the macroscopic behavior of systems.",
        "Electromagnetism": "Electromagnetism studies electric charge, electric fields, magnetic fields, electromagnetic forces, and electromagnetic waves.",
        "Quantum mechanics": "Quantum mechanics explains the behavior of matter and radiation at atomic and subatomic scales using quantization, wavefunctions, and probabilities.",
        "Relativity": "Relativity describes space, time, gravity, energy, and motion at high speeds and in strong gravitational fields.",
        "Optics": "Optics studies light, reflection, refraction, lenses, imaging, and wave behavior.",
        "Particle physics": "Particle physics investigates elementary particles and the fundamental interactions of nature."
    }

    def __init__(self, config: RetrievalConfig):
        self.config = config
        self.embeddings = LocalHashEmbeddings(dimension=256)
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""],
        )

    def build_index(self) -> FAISS:
        index_dir = Path(self.config.faiss_index_path)
        index_marker = index_dir / "index.faiss"
        metadata_path = index_dir / "index_metadata.json"
        expected_metadata = {
            "corpus": "physics_topics_v3",
            "embedding": "local_hash_v2",
            "num_articles": max(10, min(self.config.num_articles, len(self.PHYSICS_SUMMARIES))),
        }

        if index_marker.exists() and metadata_path.exists():
            stored_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            if stored_metadata == expected_metadata:
                logger.info("[Stage 2] Loading existing FAISS index from %s", index_dir)
                return FAISS.load_local(
                    str(index_dir),
                    self.embeddings,
                    allow_dangerous_deserialization=True,
                )

        raw_docs = self._load_physics_documents()
        if not raw_docs:
            raise RuntimeError("Unable to build the physics retrieval corpus.")

        chunks = self.splitter.split_documents(raw_docs)
        for chunk_id, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = chunk_id

        logger.info("[Stage 2] Building FAISS vector store with %s chunks", len(chunks))
        vectorstore = FAISS.from_documents(chunks, self.embeddings)

        index_dir.mkdir(parents=True, exist_ok=True)
        vectorstore.save_local(str(index_dir))
        metadata_path.write_text(json.dumps(expected_metadata, indent=2), encoding="utf-8")
        logger.info("[Stage 2] FAISS index saved to %s", index_dir)
        return vectorstore

    def retrieve(self) -> list[dict[str, Any]]:
        vectorstore = self.build_index()
        query = self._read_query()
        logger.info("[Stage 2] Query: %s", query)

        results = vectorstore.similarity_search_with_score(query, k=self.config.top_k)

        retrieved_chunks: list[dict[str, Any]] = []
        for rank, (doc, score) in enumerate(results, start=1):
            retrieved_chunks.append(
                {
                    "content": doc.page_content,
                    "source": doc.metadata.get("source", "unknown"),
                    "chunk_id": doc.metadata.get("chunk_id"),
                    "retrieval_rank": rank,
                    "score": float(score),
                    "metadata": doc.metadata,
                }
            )

        self.config.retrieval_output_path.parent.mkdir(parents=True, exist_ok=True)
        self.config.retrieval_output_path.write_text(
            json.dumps(retrieved_chunks, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        logger.info("[Stage 2] Saved %s retrieved chunks to %s", len(retrieved_chunks), self.config.retrieval_output_path)
        return retrieved_chunks

    def _load_physics_documents(self) -> list[Document]:
        raw_docs: list[Document] = []
        topic_titles = list(self.PHYSICS_SUMMARIES.keys())[: max(10, min(self.config.num_articles, len(self.PHYSICS_SUMMARIES)))]
        session = requests.Session()

        logger.info("[Stage 2] Building focused physics corpus from %s topics", len(topic_titles))

        for title in tqdm(topic_titles, desc="Loading physics topics"):
            summary_text = self.PHYSICS_SUMMARIES[title]
            content = f"{title}\n\n{summary_text}"
            loader = "local_summary"

            try:
                response = session.get(
                    "https://en.wikipedia.org/api/rest_v1/page/summary/" + title.replace(" ", "_"),
                    timeout=10,
                )
                if response.status_code == 200:
                    data = response.json()
                    text_parts = [
                        (data.get("title") or "").strip(),
                        (data.get("description") or "").strip(),
                        (data.get("extract") or "").strip(),
                    ]
                    api_content = "\n\n".join(part for part in text_parts if part)
                    if api_content:
                        content = api_content
                        loader = "wikipedia_api"
            except Exception as exc:
                logger.warning("[Stage 2] Using local fallback summary for %s: %s", title, exc)

            raw_docs.append(
                Document(
                    page_content=content,
                    metadata={
                        "source": title,
                        "loader": loader,
                        "topic_area": "physics",
                    },
                )
            )

        return raw_docs

    def _read_query(self) -> str:
        query = self.config.query_artifact_path.read_text(encoding="utf-8").strip()
        if not query:
            raise ValueError("Query artifact is empty.")
        return query


In [ ]:
base_config = ConfigurationManager().get_retrieval_layer_config()
config = RetrievalConfig(
    root_dir=base_config.root_dir,
    query_artifact_path=base_config.query_artifact_path,
    faiss_index_path=base_config.faiss_index_path,
    retrieval_output_path=base_config.retrieval_output_path,
    embedding_model=base_config.embedding_model,
    top_k=base_config.top_k,
    chunk_size=base_config.chunk_size,
    chunk_overlap=base_config.chunk_overlap,
    num_articles=15,
    min_score_threshold=base_config.min_score_threshold,
)

retriever = RetrievalLayer(config)
results = retriever.retrieve()
results[:3]
